# 01 — Visualisation et statistiques des datasets endoscopiques

Ce notebook (Étape C) est prêt pour **« Tout exécuter »** sur Google Colab ou en local :
1. Monte Google Drive (sur Colab) et localise le dépôt `endo-depth`.
2. Charge ton dataset (`c3vd` ou `simcol3d` depuis Drive/local). *Si aucun dataset réel n'est encore téléchargé, crée automatiquement une démo synthétique au format exact C3VD pour tester le pipeline.*
3. Calcule les statistiques (profondeur en mm, pixels valides, reflets spéculaires) et les sauvegarde en JSON.
4. Affiche images RGB, cartes de profondeur, normales de surface et masques de reflets.

In [ ]:
import importlib.util
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# === PARAMÈTRES UTILISATEUR ===
REPO_URL = ""  # Optionnel sur Colab : URL GitHub de ton fork endo-depth si tu veux cloner auto
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/endo-depth")
DATASET_NAME = "c3vd"  # "c3vd" ou "simcol3d"
IMAGE_SIZE = (256, 256)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    if Path("/content/endo-depth/src").is_dir():
        PROJECT_ROOT = Path("/content/endo-depth")
    elif (DRIVE_PROJECT_DIR / "src").is_dir():
        PROJECT_ROOT = DRIVE_PROJECT_DIR
    elif REPO_URL:
        !git clone {REPO_URL} /content/endo-depth
        PROJECT_ROOT = Path("/content/endo-depth")
    else:
        raise RuntimeError(
            "Dépôt endo-depth introuvable sur Colab. Place le dossier dans "
            "MyDrive/endo-depth ou renseigne REPO_URL."
        )
    DATA_BASE = DRIVE_PROJECT_DIR / "data"
    LOCAL_DATA_BASE = Path("/content/data")
    RESULTS_DIR = DRIVE_PROJECT_DIR / "results"
else:
    cwd = Path.cwd()
    PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
    DATA_BASE = PROJECT_ROOT / "data"
    LOCAL_DATA_BASE = PROJECT_ROOT / "data"
    RESULTS_DIR = PROJECT_ROOT / "results"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

missing = [p for p in ["yaml", "cv2"] if importlib.util.find_spec(p) is None]
if missing:
    %pip install -q pyyaml opencv-python-headless

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_BASE    :", DATA_BASE)
print("RESULTS_DIR  :", RESULTS_DIR)

In [ ]:
from src.datasets import C3VDDataset, SimCol3DDataset, resolve_dataset_dir

dataset_dir, DATASET_NAME, using_demo = resolve_dataset_dir(
    DATASET_NAME,
    drive_data_base=DATA_BASE,
    local_data_base=LOCAL_DATA_BASE,
    demo_fallback_dir=RESULTS_DIR / "_demo_c3vd",
)
print("Dossier lu par les loaders :", dataset_dir)

DatasetCls = C3VDDataset if DATASET_NAME == "c3vd" else SimCol3DDataset
full_ds = DatasetCls(dataset_dir, split="all", image_size=IMAGE_SIZE)
train_ds = DatasetCls(dataset_dir, split="train", image_size=IMAGE_SIZE)
val_ds = DatasetCls(dataset_dir, split="val", image_size=IMAGE_SIZE)
test_ds = DatasetCls(dataset_dir, split="test", image_size=IMAGE_SIZE)

print(f"Dataset chargé ({'DÉMO SYNTHÉTIQUE' if using_demo else 'RÉEL'}) : {DATASET_NAME}")
print(f"Total frames : {len(full_ds)} | train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

In [ ]:
import json
import numpy as np

max_samples_for_stats = min(len(full_ds), 100)
indices = np.linspace(0, len(full_ds) - 1, max_samples_for_stats, dtype=int)

all_depths = []
valid_ratios = []
specular_ratios = []
has_normals_count = 0

for idx in indices:
    sample = full_ds[int(idx)]
    mask = sample["mask"].squeeze(0).numpy()
    depth = sample["depth"].squeeze(0).numpy()
    spec = sample["specular_mask"].squeeze(0).numpy()

    valid_ratios.append(float(mask.mean()))
    specular_ratios.append(float((spec & mask).sum() / max(1, mask.sum())))
    if sample["has_normals"]:
        has_normals_count += 1
    if mask.any():
        # Échantillonnage pour limiter la RAM sur de gros datasets
        valid_vals = depth[mask]
        step = max(1, len(valid_vals) // 2000)
        all_depths.append(valid_vals[::step])

depth_concat = np.concatenate(all_depths) if all_depths else np.array([0.0])
stats = {
    "dataset": DATASET_NAME,
    "is_synthetic_demo": using_demo,
    "total_frames": len(full_ds),
    "splits": {"train": len(train_ds), "val": len(val_ds), "test": len(test_ds)},
    "sampled_frames": int(max_samples_for_stats),
    "has_normals_ratio": round(has_normals_count / max(1, len(indices)), 4),
    "mean_valid_pixel_ratio": round(float(np.mean(valid_ratios)), 4),
    "mean_specular_pixel_ratio": round(float(np.mean(specular_ratios)), 4),
    "depth_mm": {
        "min": round(float(depth_concat.min()), 2),
        "median": round(float(np.median(depth_concat)), 2),
        "mean": round(float(depth_concat.mean()), 2),
        "max": round(float(depth_concat.max()), 2),
        "std": round(float(depth_concat.std()), 2),
    },
}

stats_path = RESULTS_DIR / "dataset_stats.json"
with stats_path.open("w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)

print(json.dumps(stats, indent=2, ensure_ascii=False))
print(f"Statistiques sauvegardées dans : {stats_path}")

In [ ]:
import matplotlib.pyplot as plt

num_show = min(3, len(full_ds))
show_indices = np.linspace(0, len(full_ds) - 1, num_show, dtype=int)

fig, axes = plt.subplots(num_show, 4, figsize=(16, 4 * num_show))
if num_show == 1:
    axes = axes[np.newaxis, :]

for row, idx in enumerate(show_indices):
    sample = full_ds[int(idx)]
    rgb = sample["image"].permute(1, 2, 0).numpy()
    depth = sample["depth"].squeeze(0).numpy()
    mask = sample["mask"].squeeze(0).numpy()
    spec = sample["specular_mask"].squeeze(0).numpy()
    normals = sample["normals"].permute(1, 2, 0).numpy()

    axes[row, 0].imshow(rgb)
    axes[row, 0].set_title(f"{sample['sequence']}/{sample['frame_id']}\nRGB")
    axes[row, 0].axis("off")

    depth_vis = np.ma.masked_where(~mask, depth)
    im = axes[row, 1].imshow(depth_vis, cmap="plasma")
    axes[row, 1].set_title("Profondeur GT (mm)")
    axes[row, 1].axis("off")
    plt.colorbar(im, ax=axes[row, 1], fraction=0.046, pad=0.04)

    if sample["has_normals"]:
        normals_vis = np.clip((normals + 1.0) * 0.5, 0.0, 1.0)
        axes[row, 2].imshow(normals_vis)
        axes[row, 2].set_title("Normales GT ([-1,1] -> RGB)")
    else:
        axes[row, 2].text(0.5, 0.5, "Normales non disponibles", ha="center", va="center")
        axes[row, 2].set_title("Normales")
    axes[row, 2].axis("off")

    axes[row, 3].imshow(rgb)
    axes[row, 3].imshow(np.ma.masked_where(~spec, spec), cmap="spring", alpha=0.7)
    axes[row, 3].set_title(f"Reflets détectés ({spec.mean()*100:.2f}%)")
    axes[row, 3].axis("off")

plt.tight_layout()
plt.show()

# Histogramme des profondeurs valides
plt.figure(figsize=(8, 4))
plt.hist(depth_concat, bins=40, color="teal", edgecolor="black", alpha=0.8)
plt.xlabel("Profondeur (mm)")
plt.ylabel("Fréquence d'échantillons")
plt.title(f"Distribution des profondeurs — {DATASET_NAME}")
plt.grid(alpha=0.3)
plt.show()